In [8]:
import numpy as np
import pyvista as pv
from pyau3d.files import TopFile, PartFile, ModeFile
from pyau3d.utils import PltFileUtils, NodFileUtils, AnimSmartStream, GrpFileUtils
from pyau3d.pv.loaders import GrpFileVTK
from pyau3d.pv.loader.grpfilevtk import _grp2pv
from pyau3d.pv.vtktool.pdata import polydata
import vtk
import matplotlib.pyplot as plt
from matplotlib import cm, ticker
from pyau3d.constants import Constants
from scipy.signal import hilbert
# Enable interactive backend
#%matplotlib widget

# specify group for animation file
# group = 2 corresponds to cylinder wall - could double check in .an02 file
# group = 1 corresponds to left surface
RE = 50
GROUP = 1
ianimgrp = GROUP

cfd_dir = f"C:/Users/User/Git/global_stability/2d_cylinder_12625_Re{RE}_unsteady"
nod_file = f"cylinder.nod"
plt_file = "cylinder.plt"
grp_file =f"cylinder.grp{GROUP:02}"
an_file = f"cylinder.an{GROUP:02}"
part_file = "cylinder.part"
top_file = "cylinder.top"
# specify file paths

path2nod = cfd_dir + "/" + nod_file
path2plt = cfd_dir  + "/" +  plt_file
path2grp = cfd_dir  + "/" +  grp_file
path2an = cfd_dir  + "/" +  an_file
path2part = cfd_dir  + "/" +  part_file
path2top = cfd_dir  + "/" +  top_file
 
mesh = PltFileUtils(path2plt)
nod = NodFileUtils(path2nod)
grp = GrpFileUtils(path2grp,GROUP)
groupfile = GrpFileVTK(path2grp,GROUP)
anim_vars = nod.ivars[GROUP - 1]
part_file = PartFile(path2part)
top = TopFile(path2top)

anim = AnimSmartStream(
    path2an,
    anim_vars=anim_vars,
    part=part_file,
    grp=grp
)
 

In [21]:
# define delta t
deltat = top["dtstr"] / Constants()["u"]

start = 1 # start of frame
end = 10000  # end of frame
interval = 50
iframe = np.arange(start,end,interval)

# oscillation frequency is 5 Hz -> T = 0.2s
# capture 0s, 0.05, 0.1, 0.15, 0.20 for a whole period
# set interval as 50

In [20]:
locrho = np.searchsorted(nod.ivars[ianimgrp - 1],4)
locU = np.searchsorted(nod.ivars[ianimgrp - 1],[5,6,7])
locp = np.searchsorted(nod.ivars[ianimgrp - 1],9)

variables = np.arange(nod.nvars[ianimgrp - 1])

# plot contours of one frame
i = 0 # frame

# frame data (i = frames)
framedata = anim.read_frame(i, variables) # redim = True,)

# convert data to polydata
mesh = groupfile.transformtopv()
mesh['rho'] = framedata[:, locrho]
mesh['U'] = framedata[:, locU]
mesh['p'] = framedata[:, locp]

# extract u,v,w from vector U (with 3 components)
mesh['u'] = mesh['U'][:, 0]
mesh['v'] = mesh['U'][:, 1]
mesh['w'] = mesh['U'][:, 2]

# Calculate the physical time for this specific frame
physical_time = i * deltat

# plot by pyvista
# Set up the plotter window
pl = pv.Plotter(window_size=[1024, 768])

# Add the surface mesh to the scene, active scalar set to 'u_velocity'
pl.add_mesh(
    mesh, 
    scalars='p', 
    cmap='viridis',          # Clean colormap for fluid dynamics
    show_edges=True,         # Set to True if you want to see the grid lines
    # scalar_bar_args={'title': 'Velocity u (m/s)'}
    scalar_bar_args={'title': 'Pressure (Pa)}'}
)

# --- ADDED: Display Frame and Physical Time HUD ---
pl.add_text(
    text=f"Frame: {i} | Time: {physical_time:.5f} s", 
    position='upper_left', 
    font_size=12, 
    color='black'
)

# Set view to 2D since it's a 2D cylinder case (XY Plane)
pl.view_xy()
pl.show_axes()
pl.enable_anti_aliasing()

# Display the plot
pl.show()

Widget(value='<iframe src="http://localhost:56850/index.html?ui=P_0x245eca7d590_10&reconnect=auto" class="pyvi…

In [7]:

# Set up the plotter window
pl = pv.Plotter(window_size=[1024, 768])

# Initialize the mesh with an early frame so add_mesh sets up the colormap scaling
initial_framedata = anim.read_frame(iframe[0], variables)
mesh['U'] = initial_framedata[:, locU]
mesh['u'] = mesh['U'][:, 0]

# Add the surface mesh to the scene, active scalar set to 'u'
pl.add_mesh(
    mesh, 
    scalars='u', 
    cmap='viridis',          # Clean colormap for fluid dynamics
    show_edges=False,        # Set to True if you want to see the grid lines
    scalar_bar_args={'title': 'Velocity u (m/s)'}
)

# Set view to 2D since it's a 2D cylinder case (XY Plane)
pl.view_xy()
pl.show_axes()
pl.enable_anti_aliasing()

# --- MODIFICATION START: Open interactive window & loop ---

# Display the initial plot window without closing it immediately
pl.show(auto_close=False, interactive_update=True)
# --- MODIFICATION START: Save to MP4 Movie ---

# Define your video file name
video_filename = f"flow_animation_re{RE}_group{GROUP:02}.gif"

# Open the movie framework (FPS=24 provides smooth playback)
# Note: If this fails due to a missing codec on Windows, change extension to ".gif"
pl.open_movie(video_filename)

print(f"Recording animation to {video_filename}... Please wait until it completes.")

# Loop through your frame range to record the flow field
for frame_idx in iframe:
    # Read data for the current frame
    framedata = anim.read_frame(frame_idx, variables)
    
    # Update the arrays in-place
    mesh['U'] = framedata[:, locU]
    mesh['u'] = mesh['U'][:, 0]
    
    # Track frame number and physical time dynamically on the video frame
    pl.add_text(f"Frame: {frame_idx} | Time: {frame_idx * deltat:.5f}s", name='hud', position='upper_left')
    
    # Update the plot mapping arrays internally
    pl.update()
    
    # Write the current state of the window into the video file
    pl.write_frame()

# Finalize the MP4 file and close the plotter safely
pl.close()
print(f"Successfully generated: {video_filename}")

Widget(value='<iframe src="http://localhost:56850/index.html?ui=P_0x245b90a4e10_1&reconnect=auto" class="pyvis…

Recording animation to flow_animation_re50_group01.gif... Please wait until it completes.
Successfully generated: flow_animation_re50_group01.gif


In [6]:
U_inf = 34.02626486 # farfield velocity (m/s)
rho_inf = 2.10E-05 # farfield density (kg/m^3)
surface_area = 1
